In [1]:
# If needed:
# %pip install -q pystac-client rasterio xarray zarr boto3


In [2]:
import os
from pathlib import Path
import boto3

TILE = "56FQF"             # e.g. "56FQF"
START_DATE = "2015-06-01"  # YYYY-MM-DD
END_DATE = "2025-12-31"    # YYYY-MM-DD

OUT_DIM = 128
BATCH_SIZE = 24
MAX_WORKERS = max(1, (os.cpu_count() or 8) - 1)

ZARR_PATH = "s2_l1c_timecube.zarr"
CACHE_DIR = Path("./cache"); CACHE_DIR.mkdir(exist_ok=True, parents=True)

AWS_REGION = "eu-central-1"
AWS_PROFILE =  "source-keys"# set to a profile name for local runs, or None on AWS IAM roles
REQUESTER_PAYS = True

BANDS = ("B01","B02","B03","B04","B05","B06","B07","B08","B8A","B09","B11","B12")
QUANTIFICATION_VALUE = 10000.0


In [3]:
import json
import datetime as dt
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

import numpy as np
from pystac_client import Client

STAC_API_URL = "https://earth-search.aws.element84.com/v1"
STAC_COLLECTION = "sentinel-2-l1c"

ASSET_KEY_FOR_BAND: Dict[str, str] = {
    "B01": "coastal",
    "B02": "blue",
    "B03": "green",
    "B04": "red",
    "B05": "rededge1",
    "B06": "rededge2",
    "B07": "rededge3",
    "B08": "nir",
    "B8A": "nir08",
    "B09": "nir09",
    "B11": "swir16",
    "B12": "swir22",
}

def parse_tile(tile: str) -> Tuple[int, str, str]:
    tile = tile.strip().upper()
    return int(tile[:2]), tile[2], tile[3:]

def normalize_href(href: str) -> str:
    # safety for historical catalog issues
    if href.startswith("s3://sentinel-s2-l2a/"):
        return "s3://sentinel-s2-l1c/" + href[len("s3://sentinel-s2-l2a/"):]
    return href

@dataclass(frozen=True)
class S2Task:
    system_index: str
    datetime_utc: str
    band_hrefs: Dict[str, str]

def discover_tasks(tile: str, start_date: str, end_date: str, cache_path: Optional[Path] = None) -> List[S2Task]:
    if cache_path and cache_path.exists():
        return [S2Task(**x) for x in json.loads(cache_path.read_text())]

    utm, lat_band, grid_square = parse_tile(tile)
    client = Client.open(STAC_API_URL)

    search = client.search(
        collections=[STAC_COLLECTION],
        datetime=f"{start_date}/{end_date}",
        query={
            "mgrs:utm_zone": {"eq": utm},
            "mgrs:latitude_band": {"eq": lat_band},
            "mgrs:grid_square": {"eq": grid_square},
        },
    )

    items = list(search.get_items())
    items.sort(key=lambda it: (it.datetime, it.id))  # deterministic ordering

    tasks: List[S2Task] = []
    for it in items:
        hrefs = {}
        for band in BANDS:
            hrefs[band] = normalize_href(it.assets[ASSET_KEY_FOR_BAND[band]].href)
        tasks.append(S2Task(
            system_index=it.id,
            datetime_utc=it.datetime.replace(tzinfo=dt.timezone.utc).isoformat().replace("+00:00","Z"),
            band_hrefs=hrefs
        ))

    if cache_path:
        cache_path.write_text(json.dumps([t.__dict__ for t in tasks], indent=2))
    return tasks

tasks = discover_tasks(TILE, START_DATE, END_DATE, cache_path=CACHE_DIR / f"stac_{TILE}_{START_DATE}_{END_DATE}.json")
len(tasks), tasks[0]


(623,
 S2Task(system_index='S2A_56FQF_20170205_0_L1C', datetime_utc='2017-02-05T23:11:24.455000Z', band_hrefs={'B01': 's3://sentinel-s2-l1c/tiles/56/F/QF/2017/2/5/0/B01.jp2', 'B02': 's3://sentinel-s2-l1c/tiles/56/F/QF/2017/2/5/0/B02.jp2', 'B03': 's3://sentinel-s2-l1c/tiles/56/F/QF/2017/2/5/0/B03.jp2', 'B04': 's3://sentinel-s2-l1c/tiles/56/F/QF/2017/2/5/0/B04.jp2', 'B05': 's3://sentinel-s2-l1c/tiles/56/F/QF/2017/2/5/0/B05.jp2', 'B06': 's3://sentinel-s2-l1c/tiles/56/F/QF/2017/2/5/0/B06.jp2', 'B07': 's3://sentinel-s2-l1c/tiles/56/F/QF/2017/2/5/0/B07.jp2', 'B08': 's3://sentinel-s2-l1c/tiles/56/F/QF/2017/2/5/0/B08.jp2', 'B8A': 's3://sentinel-s2-l1c/tiles/56/F/QF/2017/2/5/0/B8A.jp2', 'B09': 's3://sentinel-s2-l1c/tiles/56/F/QF/2017/2/5/0/B09.jp2', 'B11': 's3://sentinel-s2-l1c/tiles/56/F/QF/2017/2/5/0/B11.jp2', 'B12': 's3://sentinel-s2-l1c/tiles/56/F/QF/2017/2/5/0/B12.jp2'}))

In [4]:
import atexit
from concurrent.futures import ProcessPoolExecutor
from typing import Tuple, Optional, List

import boto3
import rasterio
from rasterio.session import AWSSession
from rasterio.enums import Resampling

_RIO_ENV = None

def worker_init(aws_region: str, aws_profile: Optional[str], requester_pays: bool) -> None:
    global _RIO_ENV
    boto_session = boto3.Session(profile_name=aws_profile, region_name=aws_region) if aws_profile else boto3.Session(region_name=aws_region)
    aws_sess = AWSSession(boto_session, requester_pays=requester_pays)

    _RIO_ENV = rasterio.Env(
        aws_sess,
        AWS_REGION=aws_region,
        AWS_REQUEST_PAYER="requester" if requester_pays else None,
        GDAL_DISABLE_READDIR_ON_OPEN="EMPTY_DIR",
        GDAL_HTTP_MERGE_CONSECUTIVE_RANGES="YES",
        GDAL_HTTP_MULTIPLEX="YES",
        GDAL_HTTP_VERSION="2",
        GDAL_CACHEMAX=200,
        CPL_VSIL_CURL_CACHE_SIZE=200000000,
        VSI_CACHE="TRUE",
        VSI_CACHE_SIZE=5000000,
        CPL_VSIL_CURL_ALLOWED_EXTENSIONS=".jp2,.xml,.json,.JP2",
        PROJ_NETWORK="OFF",
    )
    _RIO_ENV.__enter__()
    atexit.register(lambda: _RIO_ENV.__exit__(None, None, None))

def read_one_task(task: S2Task) -> Tuple[str, np.datetime64, np.ndarray, Optional[str]]:
    try:
        t = np.datetime64(task.datetime_utc)
        band_stack = []
        for band in BANDS:
            with rasterio.open(task.band_hrefs[band]) as src:
                arr = src.read(1, out_shape=(OUT_DIM, OUT_DIM), resampling=Resampling.average)
            band_stack.append(arr.astype(np.float32) / QUANTIFICATION_VALUE)
        cube = np.stack(band_stack, axis=0)  # (band, y, x)
        return task.system_index, t, cube, None
    except Exception as e:
        return task.system_index, np.datetime64("NaT"), np.empty((0,)), repr(e)


In [5]:
def compute_georef(task: S2Task):
     # Hard-coded region
    # REGION = "eu-central-1"

    # Proper boto3 session with explicit region
    
    boto_session = boto3.Session(profile_name=AWS_PROFILE, region_name=AWS_REGION) if AWS_PROFILE else boto3.Session(region_name=AWS_REGION)
    aws_sess = AWSSession(boto_session, requester_pays=REQUESTER_PAYS)

    with rasterio.Env(aws_sess, AWS_REGION=AWS_REGION, AWS_REQUEST_PAYER="requester", GDAL_DISABLE_READDIR_ON_OPEN="EMPTY_DIR"):
        href = task.band_hrefs.get("B02") or next(iter(task.band_hrefs.values()))
        with rasterio.open(href) as src:
            transform = src.transform
            width, height = src.width, src.height

    scale_x = width / OUT_DIM
    scale_y = height / OUT_DIM
    a = transform.a * scale_x
    e = transform.e * scale_y
    x0 = transform.c
    y0 = transform.f

    x = x0 + a * (0.5 + np.arange(OUT_DIM))
    y = y0 + e * (0.5 + np.arange(OUT_DIM))
    return x, y

x_coords, y_coords = compute_georef(tasks[0])
x_coords[:3], y_coords[:3]


(array([700388.90625, 701246.71875, 702104.53125]),
 array([4099591.09375, 4098733.28125, 4097875.46875]))

In [6]:
import shutil
from pathlib import Path

if Path(ZARR_PATH).exists():
    shutil.rmtree(ZARR_PATH)
    print("Deleted old poisoned store:", ZARR_PATH)

Deleted old poisoned store: s2_l1c_timecube.zarr


In [7]:
import xarray as xr
import zarr

def append_block(zarr_path: str, pids, times, cubes, x, y, first: bool):
    data = np.stack(cubes, axis=0)  # (time, band, y, x)
    ds = xr.Dataset(
        {"reflectance": (("time","band","y","x"), data)},
        coords={
            "time": ("time", np.array(times, dtype="datetime64[ns]")),
            "band": ("band", np.array(BANDS, dtype="U3")),
            "x": ("x", x.astype(np.float64)),
            "y": ("y", y.astype(np.float64)),
            "system_index": ("time", np.array(pids, dtype="U64")),
        },
    ).chunk({"time": min(BATCH_SIZE, len(pids)), "band": len(BANDS), "y": OUT_DIM, "x": OUT_DIM})
     # IMPORTANT: force stable time encoding for Zarr append
    ds["time"].encoding = {
        "units": "nanoseconds since 1970-01-01 00:00:00",
        "calendar": "proleptic_gregorian",
        "dtype": "int64",
    }

    if first:
        ds.to_zarr(zarr_path, mode="w", consolidated=True)
    else:
        ds.to_zarr(zarr_path, mode="a", append_dim="time", consolidated=False)

# Resume
existing = set()
if Path(ZARR_PATH).exists():
    ds_existing = xr.open_zarr(
        ZARR_PATH,
        consolidated=None,
        decode_times=False,
    )
    existing = set(str(x) for x in ds_existing["system_index"].values)

tasks_todo = [t for t in tasks if t.system_index not in existing]
print("tasks total:", len(tasks), "todo:", len(tasks_todo))

first_write = not Path(ZARR_PATH).exists()
errors = []

batch_pids, batch_times, batch_cubes = [], [], []
with ProcessPoolExecutor(max_workers=MAX_WORKERS, initializer=worker_init, initargs=(AWS_REGION, AWS_PROFILE, REQUESTER_PAYS)) as ex:
    for pid, t, cube, err in ex.map(read_one_task, tasks_todo, chunksize=1):
        if err is not None:
            errors.append((pid, err))
            continue
        batch_pids.append(pid); batch_times.append(t); batch_cubes.append(cube)
        if len(batch_pids) >= BATCH_SIZE:
            append_block(ZARR_PATH, batch_pids, batch_times, batch_cubes, x_coords, y_coords, first_write)
            first_write = False
            batch_pids, batch_times, batch_cubes = [], [], []

if batch_pids:
    append_block(ZARR_PATH, batch_pids, batch_times, batch_cubes, x_coords, y_coords, first_write)

if errors:
    (CACHE_DIR/"errors.json").write_text(json.dumps(errors[:1000], indent=2))
    print("errors:", len(errors), "wrote first 1000 to", CACHE_DIR/"errors.json")

# consolidate metadata once at the end
zarr.consolidate_metadata(ZARR_PATH)


tasks total: 623 todo: 623


/tmp/ipykernel_1875857/1590589841.py:37: UserWarning: no explicit representation of timezones available for np.datetime64
  t = np.datetime64(task.datetime_utc)
/tmp/ipykernel_1875857/1590589841.py:37: UserWarning: no explicit representation of timezones available for np.datetime64
  t = np.datetime64(task.datetime_utc)
/tmp/ipykernel_1875857/1590589841.py:37: UserWarning: no explicit representation of timezones available for np.datetime64
  t = np.datetime64(task.datetime_utc)
/tmp/ipykernel_1875857/1590589841.py:37: UserWarning: no explicit representation of timezones available for np.datetime64
  t = np.datetime64(task.datetime_utc)
/tmp/ipykernel_1875857/1590589841.py:37: UserWarning: no explicit representation of timezones available for np.datetime64
  t = np.datetime64(task.datetime_utc)
/tmp/ipykernel_1875857/1590589841.py:37: UserWarning: no explicit representation of timezones available for np.datetime64
  t = np.datetime64(task.datetime_utc)
/tmp/ipykernel_1875857/1590589841.

errors: 1 wrote first 1000 to cache/errors.json


/home/ubuntu/AI4QC2_HIGH/.venv/lib/python3.12/site-packages/zarr/core/dtype/npy/string.py:249: UnstableSpecificationWarning: The data type (FixedLengthUTF32(length=64, endianness='little')) does not have a Zarr V3 specification. That means that the representation of arrays saved with this data type may change without warning in a future version of Zarr Python. Arrays stored with this data type may be unreadable by other Zarr libraries. Use this data type at your own risk! Check https://github.com/zarr-developers/zarr-extensions/tree/main/data-types for the status of data type specifications for Zarr V3.
  v3_unstable_dtype_warning(self)
/home/ubuntu/AI4QC2_HIGH/.venv/lib/python3.12/site-packages/zarr/api/asynchronous.py:247: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(
/home/ubuntu/AI4QC2_HIGH/.venv/lib/python3.12/site-packages/zarr/co

<Group file://s2_l1c_timecube.zarr>

In [8]:
import xarray as xr
ds = xr.open_zarr(ZARR_PATH, consolidated=True)
ds


<xarray.Dataset> Size: 489MB
Dimensions:       (time: 622, band: 12, y: 128, x: 128)
Coordinates:
  * time          (time) datetime64[ns] 5kB 2017-02-05T23:11:24.455000 ... 20...
  * band          (band) <U3 144B 'B01' 'B02' 'B03' 'B04' ... 'B09' 'B11' 'B12'
  * y             (y) float64 1kB 4.1e+06 4.099e+06 ... 3.992e+06 3.991e+06
  * x             (x) float64 1kB 7.004e+05 7.012e+05 ... 8.085e+05 8.093e+05
    system_index  (time) <U64 159kB dask.array<chunksize=(24,), meta=np.ndarray>
Data variables:
    reflectance   (time, band, y, x) float32 489MB dask.array<chunksize=(24, 12, 128, 128), meta=np.ndarray>